# Notebook for measuring runtime of Hashing, bucketing and similarity value computation 

In [ ]:
import os
import sys

def find_project_root(target_folder="masteroppgave"):
    """Find the absolute path of a folder by searching upward."""
    currentdir = os.path.abspath("__file__")  # Get absolute script path
    while True:
        if os.path.basename(currentdir) == target_folder:
            return currentdir  # Found the target folder
        parentdir = os.path.dirname(currentdir)
        if parentdir == currentdir:  # Stop at filesystem root
            return None
        currentdir = parentdir  # Move one level up

# Example usage
project_root = find_project_root("masteroppgave")

if project_root:
    sys.path.append(project_root)
    print(f"Project root f^ound: {project_root}")
else:
    raise RuntimeError("Could not find 'masteroppgave' directory")

from utils.helpers.measure_similarities import *


# Runtime for computination

# GRID

## Setup

In [ ]:
MEASURE="grid_dtw_cy"
CITY="rome"
RESOLUTION=4
LAYERS=2
PARALLEL_JOBS = 8
DATA_SIZE = 50
ITERATIONS = 3

#Strategies
BUCKETING_METHOD = "loose"
TRUE_TRAJECTORIES = True

In [ ]:
if TRUE_TRAJECTORIES:
    print("running with true trajectories")
    compute_hashed_similarity_runtimes_with_bucketing_with_true_sim(measure=MEASURE, city=CITY, res=RESOLUTION, layers=LAYERS, parallel_jobs=PARALLEL_JOBS, data_size=DATA_SIZE,  iterations=ITERATIONS, bucketing_method=BUCKETING_METHOD)

else:
    print("running with hashed trajectories")
    compute_hashed_similarity_runtimes_with_bucketing(measure=MEASURE, city=CITY, res=RESOLUTION, layers=LAYERS, parallel_jobs=PARALLEL_JOBS, data_size=DATA_SIZE,  iterations=ITERATIONS, bucketing_method=BUCKETING_METHOD)

# Disk

## Setup

In [ ]:
MEASURE="disk_dtw_cy"
CITY="rome"
DIAMETER=1.5
LAYERS=5
DISKS = 40
PARALLEL_JOBS = 8
DATA_SIZE = 100
ITERATIONS = 3

#Strategies
BUCKETING_METHOD = "loose"
TRUE_TRAJECTORIES = True

In [ ]:
if TRUE_TRAJECTORIES:
    compute_hashed_similarity_runtimes_with_bucketing_with_true_sim(measure=MEASURE, city=CITY, diameter=DIAMETER, layers=LAYERS, disks=DISKS, parallel_jobs=PARALLEL_JOBS, data_size = DATA_SIZE, iterations=ITERATIONS, bucketing_method=BUCKETING_METHOD)

else:
    compute_hashed_similarity_runtimes_with_bucketing(measure=MEASURE, city=CITY, diameter=DIAMETER, layers=LAYERS, disks=DISKS, parallel_jobs=PARALLEL_JOBS, data_size = DATA_SIZE, iterations=ITERATIONS, bucketing_method=BUCKETING_METHOD)


# Code for running several combinations of the runtime parameters

# Disk

## Setup

In [ ]:
MEASURE="disk_dtw_cy"
CITY="rome"
DIAMETER_LIST = [1.5, 2]
LAYERS_LIST = [1, 2]
DISKS_LIST = [20, 30]
PARALLEL_JOBS = 8
DATA_SIZE = 50
ITERATIONS = 3

#Strategies
BUCKETING_METHOD = "loose"
TRUE_TRAJECTORIES = False

In [ ]:
import itertools
import pandas as pd
import os

# Initialize DataFrame to store results
results = pd.DataFrame()

# Generate all combinations of parameters
param_combinations = list(itertools.product(DIAMETER_LIST, LAYERS_LIST, DISKS_LIST))

# Iterate over each combination and run the function
for diameter, layers, disks in param_combinations:
    print(f"Running for Diameter: {diameter}, Layers: {layers}, Disks: {disks}")


    if TRUE_TRAJECTORIES:
        df_result = compute_hashed_similarity_runtimes_with_bucketing_with_true_sim(
            measure=MEASURE,
            city=CITY,
            diameter=diameter,
            layers=layers,
            disks=disks,
            res=RESOLUTION,
            parallel_jobs=PARALLEL_JOBS,
            data_size=DATA_SIZE,
            iterations=ITERATIONS,
            bucketing_method=BUCKETING_METHOD
        )
    else:
        df_result = compute_hashed_similarity_runtimes_with_bucketing(
            measure=MEASURE,
            city=CITY,
            diameter=diameter,
            layers=layers,
            disks=disks,
            res=RESOLUTION,
            parallel_jobs=PARALLEL_JOBS,
            data_size=DATA_SIZE,
            iterations=ITERATIONS,
            bucketing_method=BUCKETING_METHOD
        )

    # Add parameters to result DataFrame
    df_result["Diameter"] = diameter
    df_result["Layers"] = layers
    df_result["Disks"] = disks

    # Define the desired column order
    desired_order = ["Diameter", "Layers", "Disks", 
                    "Data Size", 
                    "Average Similarity Computation Time (Seconds)", 
                    "Average Hash Generation Time (Seconds)", 
                    "Average Bucket Distribution Time (Seconds)"]

    # Reorder columns
    df_result = df_result[desired_order]


    # Round computation times to 2 decimal places
    df_result[[
        "Average Similarity Computation Time (Seconds)",
        "Average Hash Generation Time (Seconds)",
        "Average Bucket Distribution Time (Seconds)"
    ]] = df_result[[
        "Average Similarity Computation Time (Seconds)",
        "Average Hash Generation Time (Seconds)",
        "Average Bucket Distribution Time (Seconds)"
    ]].round(3)


    # Append to final results DataFrame
    results = pd.concat([results, df_result], ignore_index=True)

# Save final results to a CSV file

if TRUE_TRAJECTORIES:
    file_name = f"runtimes_{CITY}_disk_d{min(DIAMETER_LIST)}-{max(DIAMETER_LIST)}_l{min(LAYERS_LIST)}-{max(LAYERS_LIST)}_nd{min(DISKS_LIST)}-{max(DISKS_LIST)}_sz{DATA_SIZE}_multiple_configs_true.csv"
else:
    file_name = f"runtimes_{CITY}_disk_d{min(DIAMETER_LIST)}-{max(DIAMETER_LIST)}_l{min(LAYERS_LIST)}-{max(LAYERS_LIST)}_nd{min(DISKS_LIST)}-{max(DISKS_LIST)}_sz{DATA_SIZE}_multiple_configs.csv"

output_path = f"../../../results_hashed/runtimes/disk/{CITY}/{file_name}"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

results.to_csv(output_path, index=False)

# Display final results
print("\nFinal Results:")
display(results)


# Grid

## Setup

In [ ]:
MEASURE="grid_dtw_cy"
CITY="rome"
RESOLUTION_LIST = [0.3, 0.5]
LAYERS_LIST = [1, 2]
PARALLEL_JOBS = 8
DATA_SIZE = 50
ITERATIONS = 3

#Strategies
BUCKETING_METHOD = "loose"
TRUE_TRAJECTORIES = False

In [ ]:
import itertools
import pandas as pd
import os

# Initialize DataFrame to store results
results = pd.DataFrame()

# Generate all combinations of parameters
param_combinations = list(itertools.product(RESOLUTION_LIST, LAYERS_LIST))

# Iterate over each combination and run the function
for resolution, layers in param_combinations:
    print(f"Running for Resolution: {resolution}, Layers: {layers}")

    if TRUE_TRAJECTORIES:
        df_result = compute_hashed_similarity_runtimes_with_bucketing_with_true_sim(
            measure=MEASURE,
            city=CITY,
            res=resolution,  # Grid uses resolution instead of diameter
            layers=layers,
            parallel_jobs=PARALLEL_JOBS,
            data_size=DATA_SIZE,
            iterations=ITERATIONS,
            bucketing_method=BUCKETING_METHOD
        )
    else:
        df_result = compute_hashed_similarity_runtimes_with_bucketing(
            measure=MEASURE,
            city=CITY,
            res=resolution,  # Grid uses resolution instead of diameter
            layers=layers,
            parallel_jobs=PARALLEL_JOBS,
            data_size=DATA_SIZE,
            iterations=ITERATIONS,
            bucketing_method=BUCKETING_METHOD
        )

    # Add parameters to result DataFrame
    df_result["Resolution"] = resolution
    df_result["Layers"] = layers

    # Define the desired column order
    desired_order = ["Resolution", "Layers", 
                     "Data Size", 
                     "Average Similarity Computation Time (Seconds)", 
                     "Average Hash Generation Time (Seconds)", 
                     "Average Bucket Distribution Time (Seconds)"]

    # Reorder columns
    df_result = df_result[desired_order]

    # Round computation times to 3 decimal places
    df_result[[
        "Average Similarity Computation Time (Seconds)",
        "Average Hash Generation Time (Seconds)",
        "Average Bucket Distribution Time (Seconds)"
    ]] = df_result[[
        "Average Similarity Computation Time (Seconds)",
        "Average Hash Generation Time (Seconds)",
        "Average Bucket Distribution Time (Seconds)"
    ]].round(3)

    # Append to final results DataFrame
    results = pd.concat([results, df_result], ignore_index=True)

# Save final results to a CSV file

if TRUE_TRAJECTORIES:
    file_name = f"runtimes_{CITY}_grid_res{min(RESOLUTION_LIST)}-{max(RESOLUTION_LIST)}_l{min(LAYERS_LIST)}-{max(LAYERS_LIST)}_sz{DATA_SIZE}_multiple_configs_true.csv"
else:
    file_name = f"runtimes_{CITY}_grid_res{min(RESOLUTION_LIST)}-{max(RESOLUTION_LIST)}_l{min(LAYERS_LIST)}-{max(LAYERS_LIST)}_sz{DATA_SIZE}_multiple_configs.csv"

output_path = f"../../../results_hashed/runtimes/grid/{CITY}/{file_name}"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

results.to_csv(output_path, index=False)

# Display final results
print("\nFinal Results:")
display(results)
